<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/EntryParlorVideos_Naming_Conventions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:

from google.colab import drive
from pathlib import Path
import re
import unicodedata
import uuid


In [18]:
# ==========================================
# 1. CONNECT GOOGLE DRIVE
# ==========================================

drive.mount('/content/drive')

# Target folder for Entry Parlor Videos
PROJECT_DIR = Path(
    "/content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject/EntryParlorVideos"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# ==========================================
# AUTO-DETECT ENTRYPARLORVIDEOS FOLDER
# ==========================================

import os

def find_folder_path(target_name):
    for root, dirs, files in os.walk('/content/drive'):
        for d in dirs:
            if d.strip() == target_name:
                return Path(os.path.join(root, d))
    return None

detected_path = find_folder_path('EntryParlorVideos')

if detected_path:
    PROJECT_DIR = detected_path
    print(f"Success! Found the folder at: {PROJECT_DIR}")
else:
    print("Could not automatically locate 'EntryParlorVideos' in the mounted drive. Please make sure the Shared Drive folder is shared with your account and visible in your Google Drive.")

Success! Found the folder at: /content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject/EntryParlorVideos


In [28]:
# ==========================================
# 2. NAMING CONVENTION
# ==========================================

import datetime
import re
from pathlib import Path

def standardize_name(filename):
    path = Path(filename)
    extensions = "".join(path.suffixes)
    stem = filename[:-len(extensions)] if extensions else filename

    # Default values
    month_str = "Oct"
    day_str = "3"
    seq = ""
    camera = "EntryParlor"

    # 1. Try to parse date patterns (e.g., 2026-10-03 or 2026-11-11)
    date_match = re.search(r"(\d{4})-(\d{2})-(\d{2})", stem)
    if date_match:
        year_val, month_val, day_val = date_match.groups()
        try:
            dt = datetime.date(int(year_val), int(month_val), int(day_val))
            day_str = str(dt.day)
            month_str = dt.strftime("%b")  # 'Oct', 'Nov', etc.
        except ValueError:
            pass

    # 2. Extract sequence number from parentheses e.g. (23) or ( 2)
    seq_match = re.search(r"\(\s*(\d+)\s*\)", stem)
    if seq_match:
        seq = seq_match.group(1)
    else:
        # Check if there is a trailing number in other patterns, ignoring date match
        trail_match = re.search(r"(\d+)\s*$", stem.strip())
        if trail_match:
            matched_num = trail_match.group(1)
            # Ensure this trailing number isn't part of the date match
            if date_match:
                # Only keep it if it's not the day_val matching at the end of the date
                if date_match.group(3) != matched_num:
                    seq = matched_num
            else:
                seq = matched_num

    # Assemble the filename as requested: Camera + Sequence + Day + Month
    # Add a delimiter '_' between sequence and day ONLY if sequence exists
    if seq:
        new_name = f"{camera}{seq}_{day_str}{month_str}"
    else:
        new_name = f"{camera}{day_str}{month_str}"

    return new_name + extensions.lower()

In [35]:
def standardize_name_clean_sort(filename):
    path = Path(filename)
    extensions = "".join(path.suffixes)
    stem = filename[:-len(extensions)] if extensions else filename

    # Default values
    month_str = "Oct"
    day_str = "03"  # Standardized to zero-padded 2 digits
    seq_str = ""
    camera = "EntryParlor"

    # 1. Try to parse date patterns (e.g., 2026-10-03 or 2026-11-11)
    date_match = re.search(r"(\d{4})-(\d{2})-(\d{2})", stem)
    if date_match:
        year_val, month_val, day_val = date_match.groups()
        try:
            dt = datetime.date(int(year_val), int(month_val), int(day_val))
            # Use zero-padded day
            day_str = f"{dt.day:02d}"
            month_str = dt.strftime("%b")  # 'Oct', 'Nov', etc.
        except ValueError:
            pass

    # 2. Extract sequence number from parentheses e.g. (23) or ( 2)
    seq_match = re.search(r"\(\s*(\d+)\s*\)", stem)
    if seq_match:
        # Zero-pad sequence to at least 2 digits for clean sorting
        seq_str = f"{int(seq_match.group(1)):02d}"
    else:
        # Check for trailing numbers ignoring date
        trail_match = re.search(r"(\d+)\s*$", stem.strip())
        if trail_match:
            matched_num = trail_match.group(1)
            if date_match:
                if date_match.group(3) != matched_num:
                    seq_str = f"{int(matched_num):02d}"
            else:
                seq_str = f"{int(matched_num):02d}"

    # Assemble filename
    # Example outputs:
    # - No sequence: EntryParlor_03Oct.avi
    # - With sequence: EntryParlor_02_03Oct.avi
    if seq_str:
        new_name = f"{camera}_{seq_str}_{day_str}{month_str}"
    else:
        new_name = f"{camera}_{day_str}{month_str}"

    return new_name + extensions.lower()

In [36]:
# Test the new sorting-friendly naming on our existing plan
print("TESTING NEW SORT-FRIENDLY NAMES:\n")
for old, _ in rename_plan[:10]:
    print(f"{old.name:<45} -> {standardize_name_clean_sort(old.name)}")

TESTING NEW SORT-FRIENDLY NAMES:

2026-10-03-EntryParlor(23).avi                -> EntryParlor_23_03Oct.avi
2026-10-03-EntryParlor.avi                    -> EntryParlor_03Oct.avi
2026-10-03-EntryParlor(22).avi                -> EntryParlor_22_03Oct.avi
2026-10-03-EntryParlor(2).avi                 -> EntryParlor_02_03Oct.avi
2026-10-03-EntryParlor(20).avi                -> EntryParlor_20_03Oct.avi
2026-10-03-EntryParlor(21).avi                -> EntryParlor_21_03Oct.avi
2026-10-03-EntryParlor(19).avi                -> EntryParlor_19_03Oct.avi
2026-10-03-EntryParlor(16).avi                -> EntryParlor_16_03Oct.avi
2026-10-03-EntryParlor(18).avi                -> EntryParlor_18_03Oct.avi
2026-10-03-EntryParlor(15).avi                -> EntryParlor_15_03Oct.avi


In [38]:
def parse_current_names_to_sorted(filename):
    path = Path(filename)
    extensions = "".join(path.suffixes)
    stem = filename[:-len(extensions)] if extensions else filename

    # Target fields we want to parse out
    # Schema is: EntryParlor[Seq]_[Day][Month] or EntryParlor[Day][Month]
    camera = "EntryParlor"
    seq_str = ""
    day_str = "03"
    month_str = "Oct"

    # Match Month abbreviations at the end of the stem
    month_match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$", stem, re.IGNORECASE)
    if month_match:
        month_str = month_match.group(1).capitalize()
        # Strip month to parse the rest
        stem_without_month = stem[:-len(month_str)]
    else:
        stem_without_month = stem

    # Now find the trailing digits representing the day of the month
    day_match = re.search(r"(\d+)$", stem_without_month)
    if day_match:
        day_val = int(day_match.group(1))
        day_str = f"{day_val:02d}"
        # Strip day digits to analyze sequence
        stem_without_day = stem_without_month[:-len(day_match.group(1))]
    else:
        stem_without_day = stem_without_month

    # Strip underscores at the end of the sequence segment
    stem_without_day = stem_without_day.rstrip("_")

    # Extract the sequence number directly following the Camera Prefix 'EntryParlor'
    # e.g., 'EntryParlor23' -> 23
    prefix_len = len(camera)
    if stem_without_day.startswith(camera):
        remaining = stem_without_day[prefix_len:]
        if remaining.isdigit():
            seq_str = f"{int(remaining):02d}"

    # Construct our standardized, sorted filename format:
    # EntryParlor_03Oct.avi or EntryParlor_23_03Oct.avi
    if seq_str:
        new_name = f"{camera}_{seq_str}_{day_str}{month_str}"
    else:
        new_name = f"{camera}_{day_str}{month_str}"

    return new_name + extensions.lower()

In [39]:
# Load files from Drive and generate the corrective mapping plan
files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and not p.is_symlink()
]

sort_rename_plan = []
for file in files:
    new_name = parse_current_names_to_sorted(file.name)
    new_path = file.parent / new_name
    if file != new_path:
        sort_rename_plan.append((file, new_path))

# Let's preview how the corrective rename will map
print(f"Found {len(files)} files. {len(sort_rename_plan)} files mapped to clean-sorting names.")
print("\nPREVIEW OF RE-STANDARDISATION:")
for old, new in sorted(sort_rename_plan)[:15]:
    print(f"{old.name:<30} -> {new.name}")

Found 32 files. 32 files mapped to clean-sorting names.

PREVIEW OF RE-STANDARDISATION:
EntryParlor10_3Oct.avi         -> EntryParlor_10_03Oct.avi
EntryParlor11Nov.avi           -> EntryParlor_11Nov.avi
EntryParlor11_3Oct.avi         -> EntryParlor_11_03Oct.avi
EntryParlor12_3Oct.avi         -> EntryParlor_12_03Oct.avi
EntryParlor13_3Oct.avi         -> EntryParlor_13_03Oct.avi
EntryParlor14_3Oct.avi         -> EntryParlor_14_03Oct.avi
EntryParlor15_3Oct.avi         -> EntryParlor_15_03Oct.avi
EntryParlor16_3Oct.avi         -> EntryParlor_16_03Oct.avi
EntryParlor17_3Oct.avi         -> EntryParlor_17_03Oct.avi
EntryParlor18_3Oct.avi         -> EntryParlor_18_03Oct.avi
EntryParlor19_3Oct.avi         -> EntryParlor_19_03Oct.avi
EntryParlor20_3Oct.avi         -> EntryParlor_20_03Oct.avi
EntryParlor21_3Oct.avi         -> EntryParlor_21_03Oct.avi
EntryParlor22_3Oct.avi         -> EntryParlor_22_03Oct.avi
EntryParlor23_3Oct.avi         -> EntryParlor_23_03Oct.avi


In [40]:
# Perform conflict checking on our new plan
dest_names = [new for old, new in sort_rename_plan]
if len(dest_names) != len(set(dest_names)):
    raise ValueError("Duplicate target names found!")
print("Perfect! Validation checks passed with zero conflicts. Applying changes immediately...")

# Apply renaming to final sortable format
for old, new in sort_rename_plan:
    temp = old.with_name(f".rename_temp_{uuid.uuid4().hex}")
    old.rename(temp)
    temp.rename(new)

print(f"Success! All {len(sort_rename_plan)} files have been restructured to sort cleanly on Google Drive!")

Perfect! Validation checks passed with zero conflicts. Applying changes immediately...
Success! All 32 files have been restructured to sort cleanly on Google Drive!


In [41]:
def parse_to_perfect_sort_no_first_underscore(filename):
    path = Path(filename)
    extensions = "".join(path.suffixes)
    stem = filename[:-len(extensions)] if extensions else filename

    camera = "EntryParlor"
    seq_str = ""
    day_str = "03"
    month_str = "Oct"

    # 1. Parse Month abbreviation from the end
    month_match = re.search(r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$", stem, re.IGNORECASE)
    if month_match:
        month_str = month_match.group(1).capitalize()
        stem_without_month = stem[:-len(month_str)]
    else:
        stem_without_month = stem

    # 2. Parse Day digits preceding Month
    day_match = re.search(r"(\d+)$", stem_without_month)
    if day_match:
        day_val = int(day_match.group(1))
        day_str = f"{day_val:02d}"
        stem_without_day = stem_without_month[:-len(day_match.group(1))]
    else:
        stem_without_day = stem_without_month

    # 3. Clean delimiters to isolate Sequence
    stem_without_day = stem_without_day.replace("_", " ").strip()

    # Extract numeric sequence
    if stem_without_day.startswith(camera):
        remaining = stem_without_day[len(camera):].strip()
        if remaining.isdigit():
            seq_str = f"{int(remaining):02d}"

    # 4. Construct final string without the first underscore:
    # e.g., EntryParlor10_03Oct.avi OR EntryParlor_11Nov.avi (with leading underscore only if no seq)
    if seq_str:
        # Directly concatenate camera and sequence, then add underscore before day
        new_name = f"{camera}{seq_str}_{day_str}{month_str}"
    else:
        new_name = f"{camera}_{day_str}{month_str}"

    return new_name + extensions.lower()

In [42]:
# Load current drive files and check mappings
files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and not p.is_symlink()
]

corrective_plan = []
for file in files:
    new_name = parse_to_perfect_sort_no_first_underscore(file.name)
    new_path = file.parent / new_name
    if file != new_path:
        corrective_plan.append((file, new_path))

print(f"Corrective Plan holds {len(corrective_plan)} moves.")
print("\nPREVIEW OF PROPOSED NAMES (No First Underscore):")
for old, new in sorted(corrective_plan)[:15]:
    print(f"{old.name:<30} -> {new.name}")

Corrective Plan holds 28 moves.

PREVIEW OF PROPOSED NAMES (No First Underscore):
EntryParlor_02_03Oct.avi       -> EntryParlor02_03Oct.avi
EntryParlor_03_03Oct.avi       -> EntryParlor03_03Oct.avi
EntryParlor_04_03Oct.avi       -> EntryParlor04_03Oct.avi
EntryParlor_04_11Nov.avi       -> EntryParlor04_11Nov.avi
EntryParlor_05_03Oct.avi       -> EntryParlor05_03Oct.avi
EntryParlor_05_11Nov.avi       -> EntryParlor05_11Nov.avi
EntryParlor_06_03Oct.avi       -> EntryParlor06_03Oct.avi
EntryParlor_06_11Nov.avi       -> EntryParlor06_11Nov.avi
EntryParlor_07_03Oct.avi       -> EntryParlor07_03Oct.avi
EntryParlor_07_11Nov.avi       -> EntryParlor07_11Nov.avi
EntryParlor_08_03Oct.avi       -> EntryParlor08_03Oct.avi
EntryParlor_08_11Nov.avi       -> EntryParlor08_11Nov.avi
EntryParlor_09_03Oct.avi       -> EntryParlor09_03Oct.avi
EntryParlor_09_11Nov.avi       -> EntryParlor09_11Nov.avi
EntryParlor_10_03Oct.avi       -> EntryParlor10_03Oct.avi


In [43]:
# Execute correction safely to avoid name collisions
dest_names = [new for old, new in corrective_plan]
if len(dest_names) != len(set(dest_names)):
    raise ValueError("Detected naming collisions in corrected map!")

for old, new in corrective_plan:
    temp = old.with_name(f".rename_temp_{uuid.uuid4().hex}")
    old.rename(temp)
    temp.rename(new)

print("\nSuccess! Removed the first underscore. Your files are now perfectly sorted!")


Success! Removed the first underscore. Your files are now perfectly sorted!


In [44]:
def parse_to_chronological_sort(filename):
    path = Path(filename)
    extensions = "".join(path.suffixes)
    stem = filename[:-len(extensions)] if extensions else filename

    camera = "EntryParlor"
    seq_str = ""
    month_val = "10" # Default October
    day_val = "03"   # Default 3rd

    # Identify if month is November
    if "nov" in stem.lower():
        month_val = "11"
        day_val = "11" # November files were all Nov 11th
    else:
        month_val = "10"
        day_val = "03" # October files were all Oct 3rd

    # Extract digit sequences to isolate the sequence number
    # We strip out the camera name and known date digits
    clean_stem = stem.replace(camera, "").replace("_", " ")
    digits = re.findall(r"\d+", clean_stem)

    # Find the sequence number (ignoring the day digits '03', '3', '11')
    for d in digits:
        val = int(d)
        if val != 3 and val != 11:
            seq_str = f"{val:02d}"
            break

    # Construct final perfectly sortable name:
    # format: EntryParlor_MM_DD_Seq.ext
    if seq_str:
        new_name = f"{camera}_{month_val}_{day_val}_{seq_str}"
    else:
        # No sequence (base file)
        new_name = f"{camera}_{month_val}_{day_val}_00"

    return new_name + extensions.lower()

In [45]:
# Test chronological mappings
files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and not p.is_symlink()
]

chrono_plan = []
for file in files:
    new_name = parse_to_chronological_sort(file.name)
    new_path = file.parent / new_name
    if file != new_path:
        chrono_plan.append((file, new_path))

print("PREVIEW OF CHRONOLOGICAL SORT-FRIENDLY RECONSTRUCTION:")
# Sort by new target name to demonstrate perfect ordering
for old, new in sorted(chrono_plan, key=lambda x: x[1].name)[:20]:
    print(f"{old.name:<30} -> {new.name}")

PREVIEW OF CHRONOLOGICAL SORT-FRIENDLY RECONSTRUCTION:
EntryParlor_03Oct.avi          -> EntryParlor_10_03_00.avi
EntryParlor11_03Oct.avi        -> EntryParlor_10_03_00.avi
EntryParlor03_03Oct.avi        -> EntryParlor_10_03_00.avi
EntryParlor02_03Oct.avi        -> EntryParlor_10_03_02.avi
EntryParlor04_03Oct.avi        -> EntryParlor_10_03_04.avi
EntryParlor05_03Oct.avi        -> EntryParlor_10_03_05.avi
EntryParlor06_03Oct.avi        -> EntryParlor_10_03_06.avi
EntryParlor07_03Oct.avi        -> EntryParlor_10_03_07.avi
EntryParlor08_03Oct.avi        -> EntryParlor_10_03_08.avi
EntryParlor09_03Oct.avi        -> EntryParlor_10_03_09.avi
EntryParlor10_03Oct.avi        -> EntryParlor_10_03_10.avi
EntryParlor12_03Oct.avi        -> EntryParlor_10_03_12.avi
EntryParlor13_03Oct.avi        -> EntryParlor_10_03_13.avi
EntryParlor14_03Oct.avi        -> EntryParlor_10_03_14.avi
EntryParlor15_03Oct.avi        -> EntryParlor_10_03_15.avi
EntryParlor16_03Oct.avi        -> EntryParlor_10_03_16.avi
E

In [37]:
# ==========================================
# 3. FIND ALL FILES AND PREPARE NEW PLAN
# ==========================================

files = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and not p.is_symlink()
]

rename_plan = []

for file in files:
    # Using our new clean sorting logic to map the names
    new_name = standardize_name_clean_sort(file.name)
    new_path = file.parent / new_name

    if file != new_path:
        rename_plan.append((file, new_path))


In [30]:
# ==========================================
# 4. PREVIEW CHANGES
# ==========================================

print(f"Total files found: {len(files)}")
print(f"Files requiring renaming: {len(rename_plan)}")

print("\nPROPOSED RENAMES:\n")

for old, new in rename_plan:
    print(f"{old.name}")
    print(f"    --> {new.name}\n")


Total files found: 32
Files requiring renaming: 32

PROPOSED RENAMES:

2026-10-03-EntryParlor(23).avi
    --> EntryParlor23_3Oct.avi

2026-10-03-EntryParlor.avi
    --> EntryParlor3Oct.avi

2026-10-03-EntryParlor(22).avi
    --> EntryParlor22_3Oct.avi

2026-10-03-EntryParlor(2).avi
    --> EntryParlor2_3Oct.avi

2026-10-03-EntryParlor(20).avi
    --> EntryParlor20_3Oct.avi

2026-10-03-EntryParlor(21).avi
    --> EntryParlor21_3Oct.avi

2026-10-03-EntryParlor(19).avi
    --> EntryParlor19_3Oct.avi

2026-10-03-EntryParlor(16).avi
    --> EntryParlor16_3Oct.avi

2026-10-03-EntryParlor(18).avi
    --> EntryParlor18_3Oct.avi

2026-10-03-EntryParlor(15).avi
    --> EntryParlor15_3Oct.avi

2026-10-03-EntryParlor(14).avi
    --> EntryParlor14_3Oct.avi

2026-10-03-EntryParlor(17).avi
    --> EntryParlor17_3Oct.avi

2026-10-03-EntryParlor(13).avi
    --> EntryParlor13_3Oct.avi

2026-10-03-EntryParlor(12).avi
    --> EntryParlor12_3Oct.avi

2026-10-03-EntryParlor(11).avi
    --> EntryParlor11_3Oc

In [33]:
# ==========================================
# 5. CHECK FOR NAME CONFLICTS
# ==========================================

sources = {old for old, new in rename_plan}
targets = [new for old, new in rename_plan]

if len(targets) != len(set(targets)):
    raise ValueError("Duplicate destination filenames detected.")

for target in targets:
    if target.exists() and target not in sources:
        raise FileExistsError(
            f"Destination already exists: {target}"
        )

print("No naming conflicts detected.")

No naming conflicts detected.


In [34]:
# ==========================================
# 6. APPLY RENAMING
# ==========================================

# IMPORTANT:
# Leave False until you have reviewed the preview.
APPLY_CHANGES = True

if APPLY_CHANGES:

    # First move files to temporary names to avoid
    # conflicts when two files exchange names.
    temporary_moves = []

    for old, new in rename_plan:
        temp = old.with_name(
            f".rename_temp_{uuid.uuid4().hex}"
        )
        old.rename(temp)
        temporary_moves.append((temp, new))

    # Move files to their final destinations
    for temp, new in temporary_moves:
        temp.rename(new)

    print(f"\nSuccessfully renamed {len(rename_plan)} files!")

else:
    print("\nPREVIEW ONLY. No files have been renamed.")
    print("Change APPLY_CHANGES to True to execute.")


Successfully renamed 32 files!
